# Supply Chain Attacks on an ML Project


## Introduction


**Story.** FinCo's fraud team trains a fraud model. Like every ML team, they did not write most of what they use:
- ⚠️ Python packages,
- ⚠️ a pretrained model from a hub,
- ⚠️ datasets,
- ⚠️ a registry that stores the model,
- ⚠️ and a pipeline that moves it all to production.

An attacker does not need to break into FinCo. It is enough to compromise **one supplier**.


<div dir="rtl" style="text-align: right;">

**داستان:**: یک شرکت مالی یک مدل یادگیری ماشین تشخص تقلب را توسعه داده است. همانند هر تیم ام‌ال دیگر، این تیم بخش اعظم کدها را خودشان ننوشته‌اند، از جمله

- ⚠️ پکیج‌های پایتونی
- ⚠️ یک مدل از پیش آموزش دیده از یک مخزن عمومی
- ⚠️ دیتاست‌ها
- ⚠️ یک رجیستری که مدل‌ها را در آن ذخیره می‌کنند
- ⚠️ و یک پایپ‌لاین که تمام موارد را به محیط عملیاتی منتقل می‌کند
</div>

<p align="center">
<img src="./Pics/Supply-Chain-Diagram.png" alt="Screenshot" width="700">
</p>


**✅ Everything here is safe.** No real malware is used. The "malicious" pickle prints one line of text, which is enough to prove that a file can run code when it is loaded.

| Part | Attack | Defense in this lab |
|:---:|:---:|:---:|
| 1 | Typosquatting and dependency confusion | Name check, one trusted source |
| 2 | A package file is replaced | Hash pinning |
| 3 | A model file runs code when loaded | Restricted loader |
| 4 | A look-alike publisher and a swapped registry file | Publisher check, signature, layered gate |
| 5 | "Are we affected?" on the day of an incident | ML bill of materials (ML-BOM) |
| 6 | Known vulnerabilities in dependencies | pip-audit |


<div dir="rtl" style="text-align: right;">

**✅ همه‌چیز در اینجا ایمن است.** هیچ بدافزار واقعی استفاده نشده است. «فایل pickle مخرب» فقط یک خط متن چاپ می‌کند، و همین برای اثبات اینکه یک فایل می‌تواند هنگام بارگذاری کد اجرا کند کافی است.

| بخش | حمله | دفاع در این آزمایشگاه |
|:---:|:---:|:---:|
| ۱ | شبیه‌سازی نام (Typosquatting) و سردرگمی وابستگی (Dependency Confusion) | بررسی نام، یک منبع مورد اعتماد |
| ۲ | جایگزینی یک فایل بسته | پین کردن هش (Hash Pinning) |
| ۳ | اجرای کد توسط یک فایل مدل هنگام بارگذاری | لودر محدودشده (Restricted Loader) |
| ۴ | ناشر شبیه‌سازی‌شده و فایل رجیستری جایگزین‌شده | بررسی ناشر، امضا، دروازهٔ لایه‌ای |
| ۵ | «آیا ما تحت تأثیر قرار گرفته‌ایم؟» در روز یک حادثه | فهرست مواد نرم‌افزاری یادگیری ماشین (ML-BOM) |
| ۶ | آسیب‌پذیری‌های شناخته‌شده در وابستگی‌ها | pip-audit |
</div>


**Learning goals.** After this lab you can

- explain how each attack works,
- say which control stops it,
- and say what that control does **not** cover.

<div dir="rtl" style="text-align: right;">

**اهداف آموزشی**: پس از این جلسه شما باید موارد زیر را انجام بدهید

- توضیح دهید که هر حمله چگونه کار می‌کند.
- بدانید که چگونه برای هر حمله، چاره‌ای بیاندیشیم.
- و اینکه تدابیر ما برای مقابله با حملات چه کارهایی را برایمان تضمین نمی‌کنند.
</div>

In [ ]:
import hashlib, hmac, io, json, pickle, pickletools, warnings
from importlib import metadata
from pathlib import Path
import os

import numpy as np
from packaging.version import Version
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

In [ ]:
WORK = Path("supply_chain_lab_work")
os.makedirs(WORK, exist_ok=True)

print("Work folder created!")

## Data

<div dir="rtl" style="text-align: right;">

همانند جلسه قبل، یک مدل رگرسیون لجستیک می‌سازیم و دیتاست خود را به سه بخش، آموزش، طلایی و تست تقسیم می‌کنیم.
</div>

In [ ]:
feature_names = ["amount", "hour", "merchant_risk", "device_risk", "country_risk",
                 "velocity", "account_age", "n_prev_fraud", "distance", "channel_risk"]

X, y = make_classification(n_samples=1500, n_features=10, n_informative=6, n_redundant=2,
                           weights=[0.90, 0.10], class_sep=2.0, flip_y=0.0, random_state=SEED)
X = MinMaxScaler().fit_transform(X)

X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.6, stratify=y, random_state=SEED)
X_gold, X_te, y_gold, y_te = train_test_split(X_tmp, y_tmp, test_size=0.6, stratify=y_tmp, random_state=SEED)

model = LogisticRegression(max_iter=1000, class_weight={0: 1, 1: 3}).fit(X_tr, y_tr)

def metrics(m, Xe, ye):
    p = m.predict(Xe)
    return {"recall": recall_score(ye, p, zero_division=0), "precision": precision_score(ye, p, zero_division=0)}

base = metrics(model, X_te, y_te)

print(f"train {len(y_tr)} rows | gold {len(y_gold)} rows | test {len(y_te)} rows")
print(f"clean model on the test set: recall={base['recall']:.3f}  precision={base['precision']:.3f}")

## Helper functions used in several parts


- `sha256_bytes`: a fingerprint of some bytes. Change one byte and the fingerprint changes completely.
- `edit_distance`: how many single-letter edits turn one word into another (`numpy` to `numpyy` is 1).
- `norm`: lowercase the name and convert '_' to '-'.

<div dir="rtl" style="text-align: right;">

سه تابع کمکی تعریف می‌کنیم که در ادامه قصد استفاده از آنها را داریم.
</div>

In [ ]:
def sha256_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def edit_distance(a: str, b: str) -> int:
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[-1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

def norm(name: str) -> str:
    return name.lower().replace("_", "-")

In [ ]:

print("edit_distance('numpy', 'numpyy') =", edit_distance("numpy", "numpyy"))
print("edit_distance('numpy', 'numpyy') =", edit_distance("numpy", "numpyyy"))
print("edit_distance('numpy', 'numpyy') =", edit_distance("numpy", "numpyyyyyyy"))
print("edit_distance('numpy', 'numpyy') =", edit_distance("numpy", "numpy"))

print("\nSee the difference in SHA\n")

print("sha256 of b'hello'      :", sha256_bytes(b"hello"))
print("sha256 of b'hello!'     :", sha256_bytes(b"hello!"))

sha256_bytes(b"hello") == sha256_bytes(b"hello!")

## Part 1. Typosquatting and dependency confusion


**Typosquatting.** An attacker publishes a package whose name is one typo away from a popular one (✅`reqeusts` and ❌`request`). Someone mistypes `pip install request` and gets the attacker's code.

**Dependency confusion.** FinCo has an *internal* package, `finco-fraud-utils`, on a private index. If the installer also looks at the **public** index and picks the highest version, an attacker only has to publish the same name publicly with version `99.0.0`. This is the mechanism behind the 2022 `torchtriton` incident on PyTorch's nightly builds. See this [🔗URL](https://pytorch.org/blog/compromised-nightly-dependency/).

<div dir="rtl" style="text-align: right;">

**شبیه‌سازی نام و سردرگمی وابستگی**: 

- در این حالت، مهاجم اقدام به انتشار پکیجی می‌نماید که اسم آن بسیار شبیه کتابخانه‌ی اصلی است. در نتیجه افراد در هنگام نصب با اشتباه وارد نمودن نام پکیج اصلی، نسخه‌ای که آلوده است را دریافت می‌نمایند.

- یک شرکت مالی کتابخانه‌ای را توسعه داده  با عنوان `finco-fraud-utils` و این کتابخانه روی یک ایندکس خصوصی قرار دارد. مهاجم می‌تواند یک نسخه از یک پکیج با نام مشابه ولی شماره بسیار بالاتر را در ایندکس عمومی منتشر نماید. حال اگر در سرورهای این موسسه، دستور
`pip install finco-fraud-utils`
زده شود، پیپ بطور خودکار به ایندکس عمومی مراجعه نموده و نسخه آلوده را خواهد گرفت.

</div>

> The below cell, checks the package names in a requirements file and flags the ones that look like typosquats.

In [ ]:
REQUIREMENTS = """\
numpy==2.0.0
pandas==2.2.0
scikit-lern==1.5.0
reqeusts==2.32.0
torch==2.3.0
finco-fraud-utils>=0.4
fast-fraud-lib==1.0
"""

POPULAR = ["numpy", "pandas", "scikit-learn", "scipy", "torch", "tensorflow",
           "transformers", "requests", "matplotlib", "joblib"]

INTERNAL = {"finco-fraud-utils"}

def parse_requirements(text):
    out = []
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        for sep in ("==", ">=", "<=", "~="):
            if sep in line:
                name, ver = line.split(sep, 1)
                out.append((norm(name), sep + ver))
                break
        else:
            out.append((norm(line), ""))
    return out

# 🚩 Gives each package a verdict
def verdict(name, max_dist=2):
    if name in POPULAR:
        return "ok"
    if name in INTERNAL:
        return "INTERNAL"
    if any(edit_distance(name, p) <= max_dist for p in POPULAR):
        return "SUSPECT"
    return "unknown"

print(f"{'package':<20}{'spec':<10}verdict")

for name, spec in parse_requirements(REQUIREMENTS):
    print(f"{name:<20}{spec:<10}{verdict(name)}")

**In prod envs:** A CI pipeline decides by the exit code of each step. If a step exits with a non-zero code, the build stops. So you turn the verdicts into a pass or fail.

In [ ]:
import sys

bad = [(n, verdict(n)) for n, _ in parse_requirements(REQUIREMENTS)
       if verdict(n) in ("SUSPECT", "unknown")]

if bad:
    print("❌ BUILD REJECTED:", bad)
    sys.exit(1)        # non-zero exit code = the CI step fails
print("✅ Name check passed")

<div dir="rtl" style="text-align: right;">

در دو سلول فوق، در خصوص حمله `شبیه‌سازی` مثال دیدیم و یک راهکار برای مقابله با آن ارائه دادیم. حال در ادامه روی `سردرگمی وابستگی` بحث خواهیم نمود و راهکاری برای مقابله با آن را نشان خواهیم داد.

</div>

>The `resolve_unsafe` funct, searches both indexes and takes the highest version, so the attacker’s fake 99.0.0 wins. But, `resolve_safe` searches only the internal index for internal package names, so the attacker’s package is never considered.

In [ ]:
#🚩 Two package indexes. The attacker has registered our internal name on the PUBLIC index with a huge version.
PUBLIC_INDEX   = {"numpy": ["2.0.0"], "finco-fraud-utils": ["99.0.0"]}   # 99.0.0 = attacker
INTERNAL_INDEX = {"finco-fraud-utils": ["0.4.1"]}

#🚩 pip with an extra index: looks everywhere and picks the highest version.
def resolve_unsafe(name):
    cands  = [(Version(v), "public")   for v in PUBLIC_INDEX.get(name, [])]
    cands += [(Version(v), "internal") for v in INTERNAL_INDEX.get(name, [])]
    ver, src = max(cands)
    return src, str(ver)

#🚩 Internal names are only looked up on the internal index.
def resolve_safe(name):
    index = INTERNAL_INDEX if name in INTERNAL else PUBLIC_INDEX
    return ("internal" if name in INTERNAL else "public"), max(index[name], key=Version)

for fn in (resolve_unsafe, resolve_safe):
    src, ver = fn("finco-fraud-utils")
    print(f"{fn.__name__:<16} ===> finco-fraud-utils {ver} from the {src.upper()} index")

In our prod env, we have something like this `GitHub-Actions` snippet

```yaml
steps:
  - uses: actions/checkout@v4
  - run: python check_names.py # 1. fails on typosquats / unknown names
  - run: pip install --require-hashes -r requirements.txt # 2. hashes must match
    env:
      PIP_INDEX_URL: https://pypi.finco.example/simple # 3. internal mirror only
  - run: pip-audit -r requirements.txt  # 4. known vulnerabilities
  ```

## Part 2. A package file is replaced: hash pinning


Even when the name and version are right, the **file** behind them can change (a compromised index, a hijacked maintainer account). If you record the SHA-256 of every file you reviewed, any replacement is caught. In real life, `pip install --require-hashes -r requirements.txt` does exactly this. 

<div dir="rtl" style="text-align: right;">

حتی وقتی که نام و ورژن درست باشه، ممکنه فایل اصلی رو تغییر داده باشند. ولی اگر 
`SHA`
هر فایلی که در ابتدای فرایند توسعه بررسی کردیم رو پین کرده باشیم، هر تغییر و جایگزینی توسط افراد متخاصم، توسط سیستم ما به دام انداخته خواهد شد. برای همین حتمن در هنگام نصب از

`pip install --require-hashes -r requirements.txt`

استفاده کنید.

</div>

In [ ]:
DOWNLOADS = WORK / "downloads"
os.makedirs(DOWNLOADS, exist_ok=True)
rng = np.random.default_rng(SEED)

package_files = {
    "numpy-2.0.0-demo.whl": rng.bytes(2000),
    "pandas-2.2.0-demo.whl": rng.bytes(2000),
    "finco_fraud_utils-0.4.1-demo.whl": rng.bytes(2000),
}


In [ ]:
package_files.items()

In [ ]:

for fname, data in package_files.items():
    (DOWNLOADS / fname).write_bytes(data)

#🚩 The "lock file": hashes recorded on the day a human reviewed these files.
lock = {fname: sha256_bytes(data) for fname, data in package_files.items()}

print("What a hashed requirements file looks like:\n")

for fname, digest in lock.items():
    print(f"  {fname.split('-')[0]}==...  --hash=sha256:{digest[:20]}...")

def verify_lock(folder: Path, lock: dict):
    result = {}
    for fname, pinned in lock.items():
        path = folder / fname
        if not path.exists():
            result[fname] = "MISSING"
        else:
            result[fname] = "ok" if sha256_bytes(path.read_bytes()) == pinned else "MISMATCH"
    return result

print("\nBefore the attack:", verify_lock(DOWNLOADS, lock))

In [ ]:
#🚩 ATTACK: same file name, same version number, different content.
(DOWNLOADS / "finco_fraud_utils-0.4.1-demo.whl").write_bytes(rng.bytes(2000))
(DOWNLOADS / "pandas-2.2.0-demo.whl").write_bytes(rng.bytes(2000))

print("After the attack: ")
for fname, status in verify_lock(DOWNLOADS, lock).items():
    print(f"  {fname:<36}{status}")

**Defense and its limits.** The hash proves "this is the exact file we reviewed". It does **not** prove that the file is safe: if you pinned the hash of an already-malicious file, you will faithfully install it. It also means the lock file itself must be protected and reviewed.

<div dir="rtl" style="text-align: right;">

**مقابله و محدودیت‌هایش:**

هش بما می‌گوید که «این فایل دقیقن چیزی است که ما آنرا مرور کردیم».
ولی هش بما نمی‌گوید که این فایل امن است. اگر هش یک فایل از پیش آلوده شده را پین کرده باشیم، براحتی در خط تولید، همین فایل آلوده وارد سیستم ما خواهد شد. 

در نتیجه فایل
`lock`
باید بشدت مورد حفاظت قرار بگیرد.

</div>

> In real prod we do like this 👇. Specify the exact hash for each platform or OS.

```txt
# Example of a hashed requirements file.

# --- requests -----------------------------------------------------------
    # Each '--hash=sha256:...' line lists one acceptable file for this
    # version. pip will accept the install only if the file it downloads
    # matches one of these hashes.
    #
    # Two hashes here = two acceptable artifacts, typically:
    #   1- the wheel for one platform/architecture, and
    #   2- the wheel for another platform, OR the source distribution.
    # This lets the same requirements.txt work on more than one OS/arch.
requests==2.31.0 \
    --hash=sha256:58cd2187c01e70e6e26505bca751777aa9f2ee0b7f4300988b709f44e013003f \
    --hash=sha256:942c5a758f98d790eaed1a29cb6eefc7ffb0d1cf7af05c3d2791656dbd6ad1e1

# --- numpy --------------------------------------------------------------
    # Same idea: multiple hashes cover multiple platform-specific wheels
    # (e.g., Linux x86_64, macOS ARM64, Windows) and/or the sdist.
    # If you only ever install on ONE platform, one hash is sufficient.
numpy==1.24.3 \
    --hash=sha256:3c1104d3c036fb81ab923f50753648d9c5e5c7f9c1f4c9a5e4e4b5b5b5b5b5 \
    --hash=sha256:7c6c6e6e6e6e6e6e6e6e6e6e6e6e6e6e6e6e6e6e6e6e6e6e6e6e6e6e6e6e6e6e
    ```

## Part 3. A model file that runs code when you load it


Many model files (pickle, joblib, `torch.save`) are not tables of numbers. They are **instructions for rebuilding Python objects**, and a file can instruct the loader to call any function. So *loading a model file is the same as running a program from its author*.

<div dir="rtl" style="text-align: right;">

بسیاری از فایل‌های مدل‌های مورد استفاده توسط ما، جدولی از اعداد معمولی و وزن‌ها نیستند. این فایل‌ها در واقع دستور‌العملی برای بازسازی مجدد یک آبجکت پایتونی می‌باشند و یک فایل می‌تونه به لودر طوری دستور بده که هر تابعی رو فراخوانی کند.

در نتیجه بارگذاری یک مدل می‌تواند معادل با اجرای یک برنامه‌ی مخرب از سمت نویسنده‌ی آن باشد.

</div>


> Below, our "malicious" file only asks Python to call `print`. A real attacker would ask for something else. This is the safe way to prove the mechanism.

In [ ]:
#🚩 Harmless stand-in for a malicious payload.
class Proof:
    def __reduce__(self):
        return (print, ("   >>>⚠️ THIS LINE WAS PRINTED BY THE FILE while loading, not by our code",))

proof_bytes = pickle.dumps(Proof())

(WORK / "downloaded_model.pkl").write_bytes(proof_bytes)

print("Loading the downloaded file with plain pickle:")

pickle.loads((WORK / "downloaded_model.pkl").read_bytes())

>Notice that we never called `print` ourselves. The file did.
>
> At load time, pickle automatically calls whatever `__reduce__` said to call.

> A more dangerous example 👇. The attackers gather our env vars completly to see if there are any secrets ⚠️.

In [ ]:
class EnvProof:
    def __reduce__(self):
        return (print_env, (dict(os.environ),))

def print_env(env):
    print("   >>> ⚠️ THIS WAS READ FROM os.environ BY THE FILE WHILE LOADING:")
    for k in list(env)[:1]:          # just show ONE, not the whole environment 😎
        print(f"{k}={env[k]}")

env_proof_bytes = pickle.dumps(EnvProof())
(WORK / "downloaded_model_env.pkl").write_bytes(env_proof_bytes)

# print("Loading the downloaded file with plain pickle:")

pickle.loads((WORK / "downloaded_model_env.pkl").read_bytes())

**How attackers get access to these envs?**

> ☁️ Cloud Storage Sync and Legitimate Tools: Currently the Most Common.
>
> 🌐 HTTPS / Web APIs: The Most Basic "Channel". Simply POST the environment variables to an attacker-controlled web server or API endpoint.


> The below snippet lets you read what a pickle file would do, without ever loading (running) it.

In [ ]:
buf = io.StringIO()
pickletools.dis(proof_bytes, out=buf)
print(buf.getvalue())

## Part 4. The registry: look-alike publishers


**Who published this model?**

In the 2023 PoisonGPT demonstration, a tampered model was uploaded under a name missing one letter of the real organization. Checking the publisher name letter by letter is a cheap control. See this [🔗URL](https://thelaymansecurity.com/data-poisoning-backdoored-models/).

In [ ]:
TRUSTED_PUBLISHERS = ["finco-ml", "scikit-learn", "huggingface"]

#🚩 A funct that check the correct name of our trusted publishes. 
def check_publisher(requested: str):
    if requested in TRUSTED_PUBLISHERS:
        return "ok"
    near = [p for p in TRUSTED_PUBLISHERS if edit_distance(requested, p) <= 2]
    return f"REFUSE: not on the approved list, looks like {near}" if near else "REFUSE: not on the approved list"


LIST_OF_PUBLISHERS = ["finco-ml", "finc0-ml", "finco-mI", "random-uploader"]


for name in LIST_OF_PUBLISHERS:
    print(f"{name:<18}->", check_publisher(name))

## Part 5. An ML bill of materials (BOM): "are we affected?"


When a library or dataset is reported as compromised, the first question is: **do we use it, and in which release?** A bill of materials answers that in seconds.

<div dir="rtl" style="text-align: right;">

وقتی که یک کتابخونه یا دیتاست بعنوان 
`آسیب‌دیده`
 گزارش میشه، اولین سئوال که به ذهنم میرسه این:

آیا ما از اون استفاده کردیم؟ و در کدام شماره و انتشار؟

در اینجاست که داشتن 
`BOM`
بشدت برامون کاربردی میشه.

</div>

In [ ]:
KEY = b"demo-signing-key--in-real-life-keep-this-OUTSIDE-the-registry"
REG = WORK / "registry"
os.makedirs(REG, exist_ok=True)
MODEL_FILE = REG / "fraud_model_v1.pkl"
MANIFEST = REG / "manifest.json"
GOLD_TOL = 0.05
LAYERS = ["sha256", "hmac", "restricted-load", "golden-set", "weight-diff"]

def hmac_hex(msg: bytes, key: bytes) -> str:
    return hmac.new(key, msg, hashlib.sha256).hexdigest()

def manifest_message(man: dict) -> bytes:
    return f"{man['sha256']}|{man['golden_recall']:.6f}".encode()

def golden_recall(m) -> float:
    return recall_score(y_gold, m.predict(X_gold), zero_division=0) # type: ignore

def publish_clean():
    data = pickle.dumps(model)
    man = {"file": MODEL_FILE.name, "sha256": sha256_bytes(data), "golden_recall": golden_recall(model)}
    man["hmac"] = hmac_hex(manifest_message(man), KEY)
    MODEL_FILE.write_bytes(data)
    MANIFEST.write_text(json.dumps(man, indent=2))

publish_clean()

In [ ]:
DATA = WORK / "data"
os.makedirs(DATA, exist_ok=True)
np.save(DATA / "train_X.npy", X_tr)
np.save(DATA / "train_y.npy", y_tr)

data_lock = {f.name: sha256_bytes(f.read_bytes()) for f in sorted(DATA.iterdir())}
print("reviewed snapshot hashes:", {k: v[:12] + "..." for k, v in data_lock.items()})

> In the follwoing cell, we write the `BOM` manually to undesratnd this component. In `Production` we should use a generator tool and run it on every build.

In [ ]:
def lib_version(name):
    return metadata.version(name)

bom = {
    "bomFormat": "CycloneDX", "specVersion": "1.6", "version": 1,
    "metadata": {"component": {"type": "application", "name": "finco-fraud-pipeline"}},
    "components": [
        {"type": "machine-learning-model", "name": "fraud-lr", "version": "v1",
         "hashes": [{"alg": "SHA-256", "content": sha256_bytes(MODEL_FILE.read_bytes())}]},
        {"type": "data", "name": "train_X.npy", "hashes": [{"alg": "SHA-256", "content": data_lock["train_X.npy"]}]},
        {"type": "data", "name": "train_y.npy", "hashes": [{"alg": "SHA-256", "content": data_lock["train_y.npy"]}]},
        *[{"type": "library", "name": n, "version": lib_version(n)} for n in ["numpy", "scikit-learn", "scipy"]],
    ],
}
(WORK / "ml-bom.json").write_text(json.dumps(bom, indent=2))

print("components in the BOM:\n")

for c in bom["components"]:
    print(f"  {c['type']:<30}{c['name']:<16}{c.get('version', '')}")

> The cell below, simulates answering the question `are we affected?` or `are we exploitable?` on the day an incident is announced, using only the BOM; so no need to go search through code or dependencies by hand.

In [ ]:
def affected(bom, name=None, version=None, sha256=None):
    hits = []
    for c in bom["components"]:
        by_name = name and norm(c["name"]) == norm(name) and (version is None or c.get("version") == version)
        by_hash = sha256 and any(h["content"] == sha256 for h in c.get("hashes", []))
        if by_name or by_hash:
            hits.append(f"{c['type']}:{c['name']}")
    return hits

print("Incident 1: 'scikit-learn", lib_version("scikit-learn"), "is compromised'===>", affected(bom, "scikit-learn", lib_version("scikit-learn")) or "not affected")
print("Incident 2: 'requests is compromised'===>", affected(bom, "requests") or "not affected")
print("Incident 3: 'dataset with this hash is poisoned'===>", affected(bom, sha256=data_lock["train_y.npy"]) or "not affected") 

**Limits.** A BOM tells you **what to patch or retrain**. It does not find the attack. 

**In prod envs:** 
The BOM would be loaded into a system that’s queried on demand, and ideally some of this is automated by cross-referencing your BOMs against vulnerability feeds continuously, so you get alerted before you even ask.

## Part 6. Known vulnerabilities in dependencies: pip-audit


Everything so far was about *malicious* suppliers. Dependencies also have ordinary, already-reported vulnerabilities. `pip-audit` compares exact versions against public vulnerability databases. It needs internet access and `pip install pip-audit`.

> See one example in my cumputer 👇. Pay attention to these two commands:
>
> ` pip list --outdated` and `pip-audit`.

```powershell
PS E:\my-computer> pip list --outdated
Package         Version Latest Type
--------------- ------- ------ -----
pandas          3.1.0   3.1.1  wheel
platformdirs    4.14.3  4.14.5 wheel
python-dateutil 3.0.0   4.0.0  wheel
pyzmq           27.2.0  27.3.0 wheel
threadpoolctl   3.7.0   3.8.0  wheel
🚩 tornado         6.6     6.6.1  wheel
PS E:\my-computer> pip-audit          
Found 1 known vulnerability in 1 package
Name    Version ID              Fix Versions
------- ------- --------------- ------------
🚩 tornado 6.6     CVE-2026-126347 6.6.1
PS E:\my-computer>
```

> `pip-audit` also gives us the vulns of `pip`. 